# 01 — Build Leakage-Controlled Dataset Split

**NO MODEL TRAINING IN THIS NOTEBOOK.**

Purpose:
1. Preserve the original dataset.
2. Group known duplicate/near-duplicate image families.
3. Remove uncertain cross-class duplicate families.
4. Create explicit **Train / Validation / Test** partitions at the family level.
5. Verify that the known leakage pairs do not cross the new partitions.
6. Save a complete `split_manifest.csv`.

The notebook never deletes anything from `Cataract/Data`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, hashlib, random, json, io
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# EDIT ONLY THIS LINE if your Cataract folder has a different Drive path.
PROJECT = Path('/content/drive/MyDrive/Cataract')

SOURCE = PROJECT / 'Data'
OUTROOT = PROJECT / 'Data_Clean_LeakageControlled'
QUARANTINE = PROJECT / 'FINAL_REVISION_2026_08' / 'clean_split_quarantine'
REPORT_DIR = PROJECT / 'FINAL_REVISION_2026_08' / 'clean_split_audit'

assert SOURCE.exists(), f'Cannot find original dataset: {SOURCE}'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINE.mkdir(parents=True, exist_ok=True)

print('Original dataset:', SOURCE)
print('New cleaned dataset:', OUTROOT)

Mounted at /content/drive
Original dataset: /content/drive/MyDrive/Cataract/Data
New cleaned dataset: /content/drive/MyDrive/Cataract/Data_Clean_LeakageControlled


## Embedded audit evidence

These CSV records were produced by the prior exact-hash, perceptual-hash and SIFT/RANSAC audit.
The notebook embeds them directly so you do not have to upload a separate CSV before running.

In [2]:
NEAR_CSV = r'''test_path,test_class,matched_train_path,matched_train_class,match_type,phash_distance,sift_good,sift_inliers,sift_inlier_ratio
Test/Cataract/cat_0_1007.jpg,Cataract,Train/Cataract/cat_0_8343.jpg,Cataract,same-class near-duplicate,8,73,72,0.9863013698630136
Test/Cataract/cat_0_1022.jpg,Cataract,Train/Cataract/cat_0_5245.jpg,Cataract,same-class near-duplicate,12,45,43,0.9555555555555556
Test/Cataract/cat_0_1065.jpg,Cataract,Train/Cataract/cat_0_2231.jpg,Cataract,same-class near-duplicate,12,232,223,0.961206896551724
Test/Cataract/cat_0_1073.jpg,Cataract,Train/Cataract/cat_0_9852.jpg,Cataract,same-class near-duplicate,12,77,68,0.8831168831168831
Test/Cataract/cat_0_1074.jpg,Cataract,Train/Cataract/cat_0_9877.jpg,Cataract,same-class near-duplicate,10,82,79,0.9634146341463414
Test/Cataract/cat_0_1078.jpg,Cataract,Train/Cataract/cat_0_3509.jpg,Cataract,same-class near-duplicate,8,105,94,0.8952380952380953
Test/Cataract/cat_0_1085.jpg,Cataract,Train/Cataract/cat_0_5013.jpg,Cataract,same-class near-duplicate,12,122,110,0.9016393442622952
Test/Cataract/cat_0_1121.jpg,Cataract,Train/Cataract/cat_0_8901.jpg,Cataract,same-class near-duplicate,8,171,159,0.9298245614035088
Test/Cataract/cat_0_1136.jpg,Cataract,Train/Cataract/cat_0_6679.jpg,Cataract,same-class near-duplicate,10,210,202,0.961904761904762
Test/Cataract/cat_0_1181.jpg,Cataract,Train/Cataract/cat_0_2226.jpg,Cataract,same-class near-duplicate,12,87,77,0.8850574712643678
Test/Cataract/cat_0_1182.jpg,Cataract,Train/Cataract/cat_0_7726.jpg,Cataract,same-class near-duplicate,12,149,139,0.9328859060402684
Test/Cataract/cat_0_1208.jpg,Cataract,Train/Cataract/cat_0_2940.jpg,Cataract,same-class near-duplicate,2,143,123,0.8601398601398601
Test/Cataract/cat_0_121.jpg,Cataract,Train/Cataract/cat_0_1846.jpg,Cataract,same-class near-duplicate,8,119,112,0.9411764705882352
Test/Cataract/cat_0_1218.jpg,Cataract,Train/Cataract/cat_0_5620.jpg,Cataract,same-class near-duplicate,12,179,171,0.9553072625698324
Test/Cataract/cat_0_1280.jpg,Cataract,Train/Cataract/cat_0_2101.jpg,Cataract,same-class near-duplicate,6,56,52,0.9285714285714286
Test/Cataract/cat_0_1296.jpg,Cataract,Train/Cataract/cat_0_8098.jpg,Cataract,same-class near-duplicate,10,57,46,0.8070175438596491
Test/Cataract/cat_0_1315.jpg,Cataract,Train/Cataract/cat_0_8352.jpg,Cataract,same-class near-duplicate,8,93,87,0.935483870967742
Test/Cataract/cat_0_1328.jpg,Cataract,Train/Normal/cat_0_4602.jpg,Normal,cross-class near-duplicate / label conflict,8,114,106,0.9298245614035088
Test/Cataract/cat_0_1328.jpg,Cataract,Train/Cataract/cat_0_4270.jpg,Cataract,same-class near-duplicate,12,170,166,0.976470588235294
Test/Cataract/cat_0_1368.jpg,Cataract,Train/Cataract/cat_0_8507.jpg,Cataract,same-class near-duplicate,6,27,25,0.925925925925926
Test/Cataract/cat_0_1376.jpg,Cataract,Train/Cataract/cat_0_7310.jpg,Cataract,same-class near-duplicate,10,90,86,0.9555555555555556
Test/Cataract/cat_0_1405.jpg,Cataract,Train/Cataract/cat_0_6826.jpg,Cataract,same-class near-duplicate,12,111,98,0.8828828828828829
Test/Cataract/cat_0_1422.jpg,Cataract,Train/Cataract/cat_0_7839.jpg,Cataract,same-class near-duplicate,8,382,377,0.986910994764398
Test/Cataract/cat_0_1431.jpg,Cataract,Train/Cataract/cat_0_4551.jpg,Cataract,same-class near-duplicate,8,165,151,0.9151515151515152
Test/Cataract/cat_0_144.jpg,Cataract,Train/Cataract/cat_0_8178.jpg,Cataract,same-class near-duplicate,12,57,55,0.9649122807017544
Test/Cataract/cat_0_1450.jpg,Cataract,Train/Cataract/cat_0_8362.jpg,Cataract,same-class near-duplicate,12,54,50,0.925925925925926
Test/Cataract/cat_0_146.jpg,Cataract,Train/Cataract/cat_0_6367.jpg,Cataract,same-class near-duplicate,12,46,38,0.8260869565217391
Test/Cataract/cat_0_1461.jpg,Cataract,Train/Cataract/cat_0_8875.jpg,Cataract,same-class near-duplicate,6,624,617,0.9887820512820512
Test/Cataract/cat_0_150.jpg,Cataract,Train/Cataract/cat_0_6471.jpg,Cataract,same-class near-duplicate,10,83,77,0.927710843373494
Test/Cataract/cat_0_152.jpg,Cataract,Train/Cataract/cat_0_2303.jpg,Cataract,same-class near-duplicate,10,48,44,0.9166666666666666
Test/Cataract/cat_0_1522.jpg,Cataract,Train/Cataract/cat_0_8775.jpg,Cataract,same-class near-duplicate,8,74,68,0.918918918918919
Test/Cataract/cat_0_1553.jpg,Cataract,Train/Cataract/cat_0_7329.jpg,Cataract,same-class near-duplicate,10,132,116,0.8787878787878788
Test/Cataract/cat_0_1565.jpg,Cataract,Train/Cataract/cat_0_4174.jpg,Cataract,same-class near-duplicate,12,102,92,0.9019607843137256
Test/Cataract/cat_0_1566.jpg,Cataract,Train/Normal/cat_0_5885.jpg,Normal,cross-class near-duplicate / label conflict,12,65,59,0.9076923076923076
Test/Cataract/cat_0_1567.jpg,Cataract,Train/Cataract/cat_0_9384.jpg,Cataract,same-class near-duplicate,8,180,176,0.9777777777777776
Test/Cataract/cat_0_1587.jpg,Cataract,Train/Cataract/cat_0_9463.jpg,Cataract,same-class near-duplicate,12,271,257,0.948339483394834
Test/Cataract/cat_0_1599.jpg,Cataract,Train/Cataract/cat_0_2297.jpg,Cataract,same-class near-duplicate,12,157,147,0.9363057324840764
Test/Cataract/cat_0_1616.jpg,Cataract,Train/Cataract/cat_0_2662.jpg,Cataract,same-class near-duplicate,12,122,107,0.8770491803278688
Test/Cataract/cat_0_1629.jpg,Cataract,Train/Cataract/cat_0_2293.jpg,Cataract,same-class near-duplicate,8,43,39,0.9069767441860463
Test/Cataract/cat_0_1636.jpg,Cataract,Train/Cataract/cat_0_3760.jpg,Cataract,same-class near-duplicate,12,131,123,0.9389312977099236
Test/Cataract/cat_0_1645.jpg,Cataract,Train/Cataract/cat_0_6262.jpg,Cataract,same-class near-duplicate,10,104,102,0.9807692307692308
Test/Cataract/cat_0_1649.jpg,Cataract,Train/Cataract/cat_0_5885.jpg,Cataract,same-class near-duplicate,12,147,128,0.8707482993197279
Test/Cataract/cat_0_1657.jpg,Cataract,Train/Cataract/cat_0_4128.jpg,Cataract,same-class near-duplicate,8,39,33,0.8461538461538461
Test/Cataract/cat_0_1674.jpg,Cataract,Train/Cataract/cat_0_3702.jpg,Cataract,same-class near-duplicate,6,175,169,0.9657142857142856
Test/Cataract/cat_0_1679.jpg,Cataract,Train/Cataract/cat_0_6236.jpg,Cataract,same-class near-duplicate,12,210,194,0.923809523809524
Test/Cataract/cat_0_1692.jpg,Cataract,Train/Cataract/cat_0_8610.jpg,Cataract,same-class near-duplicate,10,188,168,0.8936170212765957
Test/Cataract/cat_0_1699.jpg,Cataract,Train/Cataract/cat_0_2892.jpg,Cataract,same-class near-duplicate,8,115,112,0.9739130434782608
Test/Cataract/cat_0_1719.jpg,Cataract,Train/Cataract/cat_0_4673.jpg,Cataract,same-class near-duplicate,12,72,72,1.0
Test/Cataract/cat_0_1721.jpg,Cataract,Train/Cataract/cat_0_4853.jpg,Cataract,same-class near-duplicate,12,538,511,0.949814126394052
Test/Cataract/cat_0_1729.jpg,Cataract,Train/Cataract/cat_0_6735.jpg,Cataract,same-class near-duplicate,12,293,287,0.9795221843003412
Test/Cataract/cat_0_1741.jpg,Cataract,Train/Cataract/cat_0_2790.jpg,Cataract,same-class near-duplicate,8,209,203,0.971291866028708
Test/Cataract/cat_0_1743.jpg,Cataract,Train/Cataract/cat_0_8868.jpg,Cataract,same-class near-duplicate,10,283,276,0.9752650176678446
Test/Cataract/cat_0_1761.jpg,Cataract,Train/Cataract/cat_0_4825.jpg,Cataract,same-class near-duplicate,12,111,104,0.9369369369369368
Test/Cataract/cat_0_185.jpg,Cataract,Train/Cataract/cat_0_4789.jpg,Cataract,same-class near-duplicate,12,116,108,0.9310344827586208
Test/Cataract/cat_0_188.jpg,Cataract,Train/Cataract/cat_0_3672.jpg,Cataract,same-class near-duplicate,8,188,168,0.8936170212765957
Test/Cataract/cat_0_197.jpg,Cataract,Train/Cataract/cat_0_6358.jpg,Cataract,same-class near-duplicate,12,121,112,0.9256198347107438
Test/Cataract/cat_0_205.jpg,Cataract,Train/Cataract/cat_0_9979.jpg,Cataract,same-class near-duplicate,12,55,48,0.8727272727272727
Test/Cataract/cat_0_222.jpg,Cataract,Train/Cataract/cat_0_7693.jpg,Cataract,same-class near-duplicate,12,101,74,0.7326732673267327
Test/Cataract/cat_0_228.jpg,Cataract,Train/Cataract/cat_0_4515.jpg,Cataract,same-class near-duplicate,12,101,92,0.9108910891089108
Test/Cataract/cat_0_250.jpg,Cataract,Train/Cataract/cat_0_9752.jpg,Cataract,same-class near-duplicate,8,117,110,0.9401709401709402
Test/Cataract/cat_0_255.jpg,Cataract,Train/Cataract/cat_0_9893.jpg,Cataract,same-class near-duplicate,10,50,45,0.9
Test/Cataract/cat_0_274.jpg,Cataract,Train/Cataract/cat_0_7451.jpg,Cataract,same-class near-duplicate,12,45,41,0.9111111111111112
Test/Cataract/cat_0_304.jpg,Cataract,Train/Cataract/cat_0_6195.jpg,Cataract,same-class near-duplicate,12,108,103,0.9537037037037036
Test/Cataract/cat_0_313.jpg,Cataract,Train/Cataract/cat_0_8556.jpg,Cataract,same-class near-duplicate,12,129,125,0.9689922480620154
Test/Cataract/cat_0_328.jpg,Cataract,Train/Cataract/cat_0_3343.jpg,Cataract,same-class near-duplicate,12,140,130,0.9285714285714286
Test/Cataract/cat_0_337.jpg,Cataract,Train/Cataract/cat_0_4775.jpg,Cataract,same-class near-duplicate,10,53,47,0.8867924528301887
Test/Cataract/cat_0_355.jpg,Cataract,Train/Cataract/cat_0_7720.jpg,Cataract,same-class near-duplicate,8,462,448,0.9696969696969696
Test/Cataract/cat_0_377.jpg,Cataract,Train/Cataract/cat_0_6562.jpg,Cataract,same-class near-duplicate,8,64,62,0.96875
Test/Cataract/cat_0_385.jpg,Cataract,Train/Cataract/cat_0_2966.jpg,Cataract,same-class near-duplicate,12,84,79,0.9404761904761904
Test/Cataract/cat_0_404.jpg,Cataract,Train/Cataract/cat_0_3514.jpg,Cataract,same-class near-duplicate,12,87,85,0.9770114942528736
Test/Cataract/cat_0_41.jpg,Cataract,Train/Cataract/cat_0_5247.jpg,Cataract,same-class near-duplicate,8,92,85,0.9239130434782608
Test/Cataract/cat_0_418.jpg,Cataract,Train/Cataract/cat_0_6228.jpg,Cataract,same-class near-duplicate,4,110,105,0.9545454545454546
Test/Cataract/cat_0_432.jpg,Cataract,Train/Cataract/cat_0_2478.jpg,Cataract,same-class near-duplicate,8,522,514,0.9846743295019156
Test/Cataract/cat_0_442.jpg,Cataract,Train/Cataract/cat_0_8228.jpg,Cataract,same-class near-duplicate,12,43,40,0.9302325581395348
Test/Cataract/cat_0_443.jpg,Cataract,Train/Cataract/cat_0_7576.jpg,Cataract,same-class near-duplicate,10,116,102,0.8793103448275862
Test/Cataract/cat_0_468.jpg,Cataract,Train/Cataract/cat_0_8639.jpg,Cataract,same-class near-duplicate,8,82,66,0.8048780487804879
Test/Cataract/cat_0_48.jpg,Cataract,Train/Cataract/cat_0_7187.jpg,Cataract,same-class near-duplicate,8,116,109,0.9396551724137931
Test/Cataract/cat_0_487.jpg,Cataract,Train/Cataract/cat_0_5344.jpg,Cataract,same-class near-duplicate,8,170,168,0.9882352941176472
Test/Cataract/cat_0_531.jpg,Cataract,Train/Cataract/cat_0_6203.jpg,Cataract,same-class near-duplicate,6,146,140,0.958904109589041
Test/Cataract/cat_0_535.jpg,Cataract,Train/Cataract/cat_0_1928.jpg,Cataract,same-class near-duplicate,12,132,122,0.9242424242424242
Test/Cataract/cat_0_569.jpg,Cataract,Train/Cataract/cat_0_9805.jpg,Cataract,same-class near-duplicate,10,109,93,0.8532110091743119
Test/Cataract/cat_0_589.jpg,Cataract,Train/Cataract/cat_0_5651.jpg,Cataract,same-class near-duplicate,12,231,211,0.9134199134199136
Test/Cataract/cat_0_592.jpg,Cataract,Train/Cataract/cat_0_2594.jpg,Cataract,same-class near-duplicate,12,108,91,0.8425925925925926
Test/Cataract/cat_0_618.jpg,Cataract,Train/Cataract/cat_0_3514.jpg,Cataract,same-class near-duplicate,10,57,53,0.9298245614035088
Test/Cataract/cat_0_621.jpg,Cataract,Train/Cataract/cat_0_7740.jpg,Cataract,same-class near-duplicate,12,531,528,0.9943502824858758
Test/Cataract/cat_0_648.jpg,Cataract,Train/Cataract/cat_0_9498.jpg,Cataract,same-class near-duplicate,12,75,66,0.88
Test/Cataract/cat_0_649.jpg,Cataract,Train/Cataract/cat_0_4363.jpg,Cataract,same-class near-duplicate,10,150,148,0.9866666666666668
Test/Cataract/cat_0_656.jpg,Cataract,Train/Cataract/cat_0_3165.jpg,Cataract,same-class near-duplicate,12,78,74,0.9487179487179488
Test/Cataract/cat_0_678.jpg,Cataract,Train/Cataract/cat_0_7463.jpg,Cataract,same-class near-duplicate,8,90,85,0.9444444444444444
Test/Cataract/cat_0_784.jpg,Cataract,Train/Cataract/cat_0_6892.jpg,Cataract,same-class near-duplicate,10,79,72,0.9113924050632912
Test/Cataract/cat_0_806.jpg,Cataract,Train/Cataract/cat_0_4977.jpg,Cataract,same-class near-duplicate,12,57,50,0.8771929824561403
Test/Cataract/cat_0_810.jpg,Cataract,Train/Cataract/cat_0_2363.jpg,Cataract,same-class near-duplicate,12,154,144,0.935064935064935
Test/Cataract/cat_0_818.jpg,Cataract,Train/Cataract/cat_0_8757.jpg,Cataract,same-class near-duplicate,12,126,120,0.9523809523809524
Test/Cataract/cat_0_85.jpg,Cataract,Train/Cataract/cat_0_7367.jpg,Cataract,same-class near-duplicate,12,194,176,0.9072164948453608
Test/Cataract/cat_0_852.jpg,Cataract,Train/Cataract/cat_0_7122.jpg,Cataract,same-class near-duplicate,10,50,43,0.86
Test/Cataract/cat_0_867.jpg,Cataract,Train/Cataract/cat_0_6048.jpg,Cataract,same-class near-duplicate,10,93,85,0.913978494623656
Test/Cataract/cat_0_874.jpg,Cataract,Train/Cataract/cat_0_5031.jpg,Cataract,same-class near-duplicate,12,229,215,0.9388646288209608
Test/Cataract/cat_0_890.jpg,Cataract,Train/Cataract/cat_0_3577.jpg,Cataract,same-class near-duplicate,8,106,100,0.9433962264150944
Test/Cataract/cat_0_892.jpg,Cataract,Train/Cataract/cat_0_3366.jpg,Cataract,same-class near-duplicate,10,36,31,0.8611111111111112
Test/Cataract/cat_0_932.jpg,Cataract,Train/Cataract/cat_0_7017.jpg,Cataract,same-class near-duplicate,12,184,180,0.9782608695652174
Test/Cataract/cat_0_956.jpg,Cataract,Train/Cataract/cat_0_9284.jpg,Cataract,same-class near-duplicate,8,186,176,0.946236559139785
Test/Cataract/cat_0_980.jpg,Cataract,Train/Cataract/cat_0_5855.jpg,Cataract,same-class near-duplicate,10,157,144,0.9171974522292994
Test/Normal/cat_0_1001.jpg,Normal,Train/Normal/cat_0_2052.jpg,Normal,same-class near-duplicate,12,89,83,0.9325842696629212
Test/Normal/cat_0_1020.jpg,Normal,Train/Normal/cat_0_2840.jpg,Normal,same-class near-duplicate,10,106,99,0.9339622641509434
Test/Normal/cat_0_1031.jpg,Normal,Train/Normal/cat_0_8515.jpg,Normal,same-class near-duplicate,12,74,66,0.8918918918918919
Test/Normal/cat_0_1041.jpg,Normal,Train/Normal/cat_0_8258.jpg,Normal,same-class near-duplicate,6,167,152,0.9101796407185628
Test/Normal/cat_0_1046.jpg,Normal,Train/Normal/cat_0_4085.jpg,Normal,same-class near-duplicate,12,133,131,0.9849624060150376
Test/Normal/cat_0_1053.jpg,Normal,Train/Normal/cat_0_2059.jpg,Normal,same-class near-duplicate,10,123,114,0.926829268292683
Test/Normal/cat_0_1069.jpg,Normal,Train/Normal/cat_0_2186.jpg,Normal,same-class near-duplicate,12,94,89,0.946808510638298
Test/Normal/cat_0_1073.jpg,Normal,Train/Normal/cat_0_8587.jpg,Normal,same-class near-duplicate,12,87,82,0.942528735632184
Test/Normal/cat_0_1086.jpg,Normal,Train/Normal/cat_0_5556.jpg,Normal,same-class near-duplicate,10,100,96,0.96
Test/Normal/cat_0_11.jpg,Normal,Train/Normal/cat_0_8350.jpg,Normal,same-class near-duplicate,12,102,97,0.9509803921568628
Test/Normal/cat_0_1109.jpg,Normal,Train/Normal/cat_0_8665.jpg,Normal,same-class near-duplicate,8,120,116,0.9666666666666668
Test/Normal/cat_0_1117.jpg,Normal,Train/Normal/cat_0_9699.jpg,Normal,same-class near-duplicate,12,69,64,0.927536231884058
Test/Normal/cat_0_112.jpg,Normal,Train/Normal/cat_0_5822.jpg,Normal,same-class near-duplicate,12,45,39,0.8666666666666667
Test/Normal/cat_0_1124.jpg,Normal,Train/Normal/cat_0_6485.jpg,Normal,same-class near-duplicate,8,68,61,0.8970588235294118
Test/Normal/cat_0_1132.jpg,Normal,Train/Normal/cat_0_7038.jpg,Normal,same-class near-duplicate,10,90,71,0.7888888888888889
Test/Normal/cat_0_1138.jpg,Normal,Train/Normal/cat_0_3920.jpg,Normal,same-class near-duplicate,12,43,42,0.9767441860465116
Test/Normal/cat_0_1147.jpg,Normal,Train/Normal/cat_0_7562.jpg,Normal,same-class near-duplicate,12,50,44,0.88
Test/Normal/cat_0_1150.jpg,Normal,Train/Normal/cat_0_5084.jpg,Normal,same-class near-duplicate,10,109,105,0.963302752293578
Test/Normal/cat_0_1166.jpg,Normal,Train/Normal/cat_0_3109.jpg,Normal,same-class near-duplicate,10,59,49,0.8305084745762712
Test/Normal/cat_0_1167.jpg,Normal,Train/Normal/cat_0_6828.jpg,Normal,same-class near-duplicate,10,162,160,0.9876543209876544
Test/Normal/cat_0_1202.jpg,Normal,Train/Normal/cat_0_8685.jpg,Normal,same-class near-duplicate,8,102,100,0.9803921568627452
Test/Normal/cat_0_1220.jpg,Normal,Train/Normal/cat_0_3321.jpg,Normal,same-class near-duplicate,10,121,113,0.9338842975206612
Test/Normal/cat_0_1225.jpg,Normal,Train/Normal/cat_0_7404.jpg,Normal,same-class near-duplicate,12,117,112,0.9572649572649572
Test/Normal/cat_0_1232.jpg,Normal,Train/Normal/cat_0_4863.jpg,Normal,same-class near-duplicate,12,90,87,0.9666666666666668
Test/Normal/cat_0_1234.jpg,Normal,Train/Normal/cat_0_2392.jpg,Normal,same-class near-duplicate,12,87,85,0.9770114942528736
Test/Normal/cat_0_1251.jpg,Normal,Train/Normal/cat_0_3942.jpg,Normal,same-class near-duplicate,6,111,104,0.9369369369369368
Test/Normal/cat_0_1256.jpg,Normal,Train/Normal/cat_0_6299.jpg,Normal,same-class near-duplicate,6,126,121,0.9603174603174603
Test/Normal/cat_0_126.jpg,Normal,Train/Normal/cat_0_7893.jpg,Normal,same-class near-duplicate,12,131,129,0.9847328244274808
Test/Normal/cat_0_1265.jpg,Normal,Train/Normal/cat_0_6624.jpg,Normal,same-class near-duplicate,6,77,75,0.974025974025974
Test/Normal/cat_0_1269.jpg,Normal,Train/Normal/cat_0_8786.jpg,Normal,same-class near-duplicate,12,134,130,0.9701492537313432
Test/Normal/cat_0_1277.jpg,Normal,Train/Normal/cat_0_8056.jpg,Normal,same-class near-duplicate,10,150,145,0.9666666666666668
Test/Normal/cat_0_1282.jpg,Normal,Train/Normal/cat_0_4865.jpg,Normal,same-class near-duplicate,8,93,88,0.946236559139785
Test/Normal/cat_0_1291.jpg,Normal,Train/Normal/cat_0_3186.jpg,Normal,same-class near-duplicate,12,82,76,0.926829268292683
Test/Normal/cat_0_1310.jpg,Normal,Train/Normal/cat_0_5328.jpg,Normal,same-class near-duplicate,12,81,76,0.9382716049382716
Test/Normal/cat_0_1315.jpg,Normal,Train/Normal/cat_0_7448.jpg,Normal,same-class near-duplicate,6,112,108,0.9642857142857144
Test/Normal/cat_0_1322.jpg,Normal,Train/Normal/cat_0_4526.jpg,Normal,same-class near-duplicate,10,107,102,0.9532710280373832
Test/Normal/cat_0_1324.jpg,Normal,Train/Normal/cat_0_3388.jpg,Normal,same-class near-duplicate,12,99,96,0.9696969696969696
Test/Normal/cat_0_1325.jpg,Normal,Train/Normal/cat_0_6566.jpg,Normal,same-class near-duplicate,10,82,76,0.926829268292683
Test/Normal/cat_0_1344.jpg,Normal,Train/Normal/cat_0_6425.jpg,Normal,same-class near-duplicate,8,85,72,0.8470588235294118
Test/Normal/cat_0_1345.jpg,Normal,Train/Normal/cat_0_2518.jpg,Normal,same-class near-duplicate,12,79,75,0.9493670886075948
Test/Normal/cat_0_1354.jpg,Normal,Train/Normal/cat_0_2255.jpg,Normal,same-class near-duplicate,12,62,61,0.9838709677419356
Test/Normal/cat_0_1361.jpg,Normal,Train/Normal/cat_0_5043.jpg,Normal,same-class near-duplicate,12,89,86,0.9662921348314608
Test/Normal/cat_0_1364.jpg,Normal,Train/Normal/cat_0_5908.jpg,Normal,same-class near-duplicate,10,149,138,0.9261744966442952
Test/Normal/cat_0_1394.jpg,Normal,Train/Normal/cat_0_8941.jpg,Normal,same-class near-duplicate,10,143,136,0.951048951048951
Test/Normal/cat_0_1397.jpg,Normal,Train/Normal/cat_0_1631.jpg,Normal,same-class near-duplicate,12,114,111,0.9736842105263158
Test/Normal/cat_0_1418.jpg,Normal,Train/Normal/cat_0_8852.jpg,Normal,same-class near-duplicate,10,131,125,0.9541984732824428
Test/Normal/cat_0_1438.jpg,Normal,Train/Normal/cat_0_7230.jpg,Normal,same-class near-duplicate,10,71,66,0.9295774647887324
Test/Normal/cat_0_1467.jpg,Normal,Train/Normal/cat_0_4543.jpg,Normal,same-class near-duplicate,12,119,115,0.9663865546218487
Test/Normal/cat_0_1478.jpg,Normal,Train/Normal/cat_0_9486.jpg,Normal,same-class near-duplicate,12,82,73,0.8902439024390244
Test/Normal/cat_0_148.jpg,Normal,Train/Normal/cat_0_3539.jpg,Normal,same-class near-duplicate,10,57,51,0.8947368421052632
Test/Normal/cat_0_152.jpg,Normal,Train/Normal/cat_0_1739.jpg,Normal,same-class near-duplicate,8,38,37,0.9736842105263158
Test/Normal/cat_0_1527.jpg,Normal,Train/Normal/cat_0_5834.jpg,Normal,same-class near-duplicate,12,87,83,0.9540229885057472
Test/Normal/cat_0_1532.jpg,Normal,Train/Normal/cat_0_8049.jpg,Normal,same-class near-duplicate,6,109,105,0.963302752293578
Test/Normal/cat_0_1538.jpg,Normal,Train/Normal/cat_0_6522.jpg,Normal,same-class near-duplicate,12,109,104,0.9541284403669724
Test/Normal/cat_0_1542.jpg,Normal,Train/Normal/cat_0_4093.jpg,Normal,same-class near-duplicate,12,265,260,0.981132075471698
Test/Normal/cat_0_1563.jpg,Normal,Train/Normal/cat_0_5382.jpg,Normal,same-class near-duplicate,6,129,117,0.9069767441860463
Test/Normal/cat_0_18.jpg,Normal,Train/Normal/cat_0_4080.jpg,Normal,same-class near-duplicate,12,78,73,0.935897435897436
Test/Normal/cat_0_214.jpg,Normal,Train/Normal/cat_0_3728.jpg,Normal,same-class near-duplicate,10,128,106,0.828125
Test/Normal/cat_0_22.jpg,Normal,Train/Normal/cat_0_5819.jpg,Normal,same-class near-duplicate,12,96,88,0.9166666666666666
Test/Normal/cat_0_242.jpg,Normal,Train/Normal/cat_0_3453.jpg,Normal,same-class near-duplicate,12,129,126,0.9767441860465116
Test/Normal/cat_0_248.jpg,Normal,Train/Normal/cat_0_2918.jpg,Normal,same-class near-duplicate,10,78,75,0.9615384615384616
Test/Normal/cat_0_250.jpg,Normal,Train/Normal/cat_0_7215.jpg,Normal,same-class near-duplicate,8,105,95,0.9047619047619048
Test/Normal/cat_0_254.jpg,Normal,Train/Normal/cat_0_9528.jpg,Normal,same-class near-duplicate,10,106,91,0.8584905660377359
Test/Normal/cat_0_256.jpg,Normal,Train/Normal/cat_0_8604.jpg,Normal,same-class near-duplicate,10,106,95,0.8962264150943396
Test/Normal/cat_0_264.jpg,Normal,Train/Normal/cat_0_5255.jpg,Normal,same-class near-duplicate,8,87,77,0.8850574712643678
Test/Normal/cat_0_269.jpg,Normal,Train/Normal/cat_0_1682.jpg,Normal,same-class near-duplicate,12,65,59,0.9076923076923076
Test/Normal/cat_0_308.jpg,Normal,Train/Normal/cat_0_2906.jpg,Normal,same-class near-duplicate,10,65,65,1.0
Test/Normal/cat_0_325.jpg,Normal,Train/Normal/cat_0_2009.jpg,Normal,same-class near-duplicate,8,111,105,0.945945945945946
Test/Normal/cat_0_329.jpg,Normal,Train/Normal/cat_0_2236.jpg,Normal,same-class near-duplicate,10,147,142,0.9659863945578232
Test/Normal/cat_0_330.jpg,Normal,Train/Normal/cat_0_8997.jpg,Normal,same-class near-duplicate,12,70,63,0.9
Test/Normal/cat_0_344.jpg,Normal,Train/Normal/cat_0_6727.jpg,Normal,same-class near-duplicate,12,177,171,0.9661016949152542
Test/Normal/cat_0_350.jpg,Normal,Train/Normal/cat_0_2658.jpg,Normal,same-class near-duplicate,8,139,137,0.9856115107913668
Test/Normal/cat_0_363.jpg,Normal,Train/Normal/cat_0_2799.jpg,Normal,same-class near-duplicate,12,89,85,0.9550561797752808
Test/Normal/cat_0_374.jpg,Normal,Train/Normal/cat_0_7701.jpg,Normal,same-class near-duplicate,12,77,71,0.922077922077922
Test/Normal/cat_0_396.jpg,Normal,Train/Normal/cat_0_4499.jpg,Normal,same-class near-duplicate,12,87,83,0.9540229885057472
Test/Normal/cat_0_415.jpg,Normal,Train/Normal/cat_0_1626.jpg,Normal,same-class near-duplicate,10,130,124,0.953846153846154
Test/Normal/cat_0_416.jpg,Normal,Train/Normal/cat_0_9416.jpg,Normal,same-class near-duplicate,8,166,160,0.963855421686747
Test/Normal/cat_0_445.jpg,Normal,Train/Normal/cat_0_8760.jpg,Normal,same-class near-duplicate,6,103,98,0.9514563106796116
Test/Normal/cat_0_457.jpg,Normal,Train/Normal/cat_0_1944.jpg,Normal,same-class near-duplicate,4,120,119,0.9916666666666668
Test/Normal/cat_0_459.jpg,Normal,Train/Normal/cat_0_4138.jpg,Normal,same-class near-duplicate,12,81,79,0.9753086419753086
Test/Normal/cat_0_479.jpg,Normal,Train/Normal/cat_0_4867.jpg,Normal,same-class near-duplicate,12,72,63,0.875
Test/Normal/cat_0_486.jpg,Normal,Train/Normal/cat_0_7474.jpg,Normal,same-class near-duplicate,8,62,58,0.935483870967742
Test/Normal/cat_0_495.jpg,Normal,Train/Normal/cat_0_7846.jpg,Normal,same-class near-duplicate,8,50,42,0.84
Test/Normal/cat_0_510.jpg,Normal,Train/Normal/cat_0_3478.jpg,Normal,same-class near-duplicate,12,103,99,0.9611650485436892
Test/Normal/cat_0_520.jpg,Normal,Train/Normal/cat_0_5240.jpg,Normal,same-class near-duplicate,12,78,63,0.8076923076923077
Test/Normal/cat_0_526.jpg,Normal,Train/Normal/cat_0_8599.jpg,Normal,same-class near-duplicate,8,99,90,0.9090909090909092
Test/Normal/cat_0_548.jpg,Normal,Train/Normal/cat_0_3144.jpg,Normal,same-class near-duplicate,8,63,58,0.9206349206349206
Test/Normal/cat_0_550.jpg,Normal,Train/Normal/cat_0_1776.jpg,Normal,same-class near-duplicate,10,150,141,0.94
Test/Normal/cat_0_565.jpg,Normal,Train/Normal/cat_0_7201.jpg,Normal,same-class near-duplicate,6,150,139,0.9266666666666666
Test/Normal/cat_0_574.jpg,Normal,Train/Normal/cat_0_4597.jpg,Normal,same-class near-duplicate,12,147,138,0.9387755102040816
Test/Normal/cat_0_575.jpg,Normal,Train/Normal/cat_0_4193.jpg,Normal,same-class near-duplicate,10,130,121,0.9307692307692308
Test/Normal/cat_0_586.jpg,Normal,Train/Normal/cat_0_1916.jpg,Normal,same-class near-duplicate,10,58,55,0.9482758620689656
Test/Normal/cat_0_611.jpg,Normal,Train/Normal/cat_0_9167.jpg,Normal,same-class near-duplicate,10,85,83,0.976470588235294
Test/Normal/cat_0_618.jpg,Normal,Train/Normal/cat_0_8152.jpg,Normal,same-class near-duplicate,4,93,87,0.935483870967742
Test/Normal/cat_0_661.jpg,Normal,Train/Normal/cat_0_7534.jpg,Normal,same-class near-duplicate,10,63,57,0.9047619047619048
Test/Normal/cat_0_664.jpg,Normal,Train/Normal/cat_0_5983.jpg,Normal,same-class near-duplicate,10,94,91,0.9680851063829788
Test/Normal/cat_0_666.jpg,Normal,Train/Normal/cat_0_5354.jpg,Normal,same-class near-duplicate,10,115,111,0.9652173913043478
Test/Normal/cat_0_672.jpg,Normal,Train/Normal/cat_0_6203.jpg,Normal,same-class near-duplicate,10,113,110,0.9734513274336284
Test/Normal/cat_0_695.jpg,Normal,Train/Normal/cat_0_5525.jpg,Normal,same-class near-duplicate,8,142,139,0.9788732394366196
Test/Normal/cat_0_7.jpg,Normal,Train/Normal/cat_0_8201.jpg,Normal,same-class near-duplicate,2,55,54,0.9818181818181818
Test/Normal/cat_0_708.jpg,Normal,Train/Normal/cat_0_5906.jpg,Normal,same-class near-duplicate,10,130,126,0.9692307692307692
Test/Normal/cat_0_717.jpg,Normal,Train/Normal/cat_0_9791.jpg,Normal,same-class near-duplicate,12,84,79,0.9404761904761904
Test/Normal/cat_0_72.jpg,Normal,Train/Normal/cat_0_5723.jpg,Normal,same-class near-duplicate,10,58,54,0.9310344827586208
Test/Normal/cat_0_725.jpg,Normal,Train/Normal/cat_0_1777.jpg,Normal,same-class near-duplicate,10,98,91,0.9285714285714286
Test/Normal/cat_0_73.jpg,Normal,Train/Normal/cat_0_3127.jpg,Normal,same-class near-duplicate,4,107,94,0.8785046728971962
Test/Normal/cat_0_732.jpg,Normal,Train/Normal/cat_0_9684.jpg,Normal,same-class near-duplicate,4,143,134,0.9370629370629372
Test/Normal/cat_0_733.jpg,Normal,Train/Normal/cat_0_4584.jpg,Normal,same-class near-duplicate,12,202,194,0.9603960396039604
Test/Normal/cat_0_740.jpg,Normal,Train/Normal/cat_0_3591.jpg,Normal,same-class near-duplicate,8,156,150,0.9615384615384616
Test/Normal/cat_0_749.jpg,Normal,Train/Normal/cat_0_9593.jpg,Normal,same-class near-duplicate,10,102,96,0.9411764705882352
Test/Normal/cat_0_75.jpg,Normal,Train/Normal/cat_0_1916.jpg,Normal,same-class near-duplicate,12,57,53,0.9298245614035088
Test/Normal/cat_0_766.jpg,Normal,Train/Normal/cat_0_5357.jpg,Normal,same-class near-duplicate,10,137,127,0.927007299270073
Test/Normal/cat_0_78.jpg,Normal,Train/Normal/cat_0_4079.jpg,Normal,same-class near-duplicate,12,87,77,0.8850574712643678
Test/Normal/cat_0_788.jpg,Normal,Train/Normal/cat_0_3074.jpg,Normal,same-class near-duplicate,6,79,78,0.9873417721518988
Test/Normal/cat_0_795.jpg,Normal,Train/Normal/cat_0_3375.jpg,Normal,same-class near-duplicate,12,126,123,0.9761904761904762
Test/Normal/cat_0_803.jpg,Normal,Train/Normal/cat_0_8578.jpg,Normal,same-class near-duplicate,10,113,106,0.9380530973451328
Test/Normal/cat_0_817.jpg,Normal,Train/Normal/cat_0_6592.jpg,Normal,same-class near-duplicate,10,69,66,0.9565217391304348
Test/Normal/cat_0_866.jpg,Normal,Train/Normal/cat_0_5638.jpg,Normal,same-class near-duplicate,10,120,115,0.9583333333333334
Test/Normal/cat_0_874.jpg,Normal,Train/Normal/cat_0_7236.jpg,Normal,same-class near-duplicate,12,50,49,0.98
Test/Normal/cat_0_879.jpg,Normal,Train/Normal/cat_0_2784.jpg,Normal,same-class near-duplicate,8,108,103,0.9537037037037036
Test/Normal/cat_0_894.jpg,Normal,Train/Normal/cat_0_3259.jpg,Normal,same-class near-duplicate,12,93,87,0.935483870967742
Test/Normal/cat_0_9.jpg,Normal,Train/Normal/cat_0_8626.jpg,Normal,same-class near-duplicate,10,250,248,0.992
Test/Normal/cat_0_923.jpg,Normal,Train/Normal/cat_0_3430.jpg,Normal,same-class near-duplicate,8,168,161,0.9583333333333334
Test/Normal/cat_0_961.jpg,Normal,Train/Normal/cat_0_7286.jpg,Normal,same-class near-duplicate,12,98,92,0.9387755102040816
Test/Normal/cat_0_967.jpg,Normal,Train/Normal/cat_0_6616.jpg,Normal,same-class near-duplicate,12,97,94,0.9690721649484536
Test/Normal/cat_0_975.jpg,Normal,Train/Normal/cat_0_2259.jpg,Normal,same-class near-duplicate,12,80,77,0.9625
Test/Normal/cat_0_981.jpg,Normal,Train/Normal/cat_0_3571.jpg,Normal,same-class near-duplicate,12,115,111,0.9652173913043478
Test/Normal/cat_0_999.jpg,Normal,Train/Normal/cat_0_3572.jpg,Normal,same-class near-duplicate,12,51,47,0.9215686274509804
Test/Not Eye/00a258a357.jpg,Not Eye,Train/Not Eye/3ad0d5554a.jpg,Not Eye,same-class near-duplicate,0,372,367,0.9865591397849462
Test/Not Eye/00bdc2d18a.jpg,Not Eye,Train/Not Eye/6a4ad2c8ea.jpg,Not Eye,same-class near-duplicate,12,104,92,0.8846153846153846
Test/Not Eye/00dd586fe7.jpg,Not Eye,Train/Not Eye/7b5411c0c2.jpg,Not Eye,same-class near-duplicate,0,229,222,0.9694323144104804
Test/Not Eye/01e889cf43.jpg,Not Eye,Train/Not Eye/5cd0548db4.jpg,Not Eye,same-class near-duplicate,2,177,172,0.9717514124293786
Test/Not Eye/02df621fdd.jpg,Not Eye,Train/Not Eye/5e5698e2f0.jpg,Not Eye,same-class near-duplicate,0,278,266,0.9568345323741008
Test/Not Eye/02f2e49039.jpg,Not Eye,Train/Not Eye/8ac8bbff12.jpg,Not Eye,same-class near-duplicate,0,225,213,0.9466666666666668
Test/Not Eye/02fa84068c.jpg,Not Eye,Train/Not Eye/46db7f90d4.jpg,Not Eye,same-class near-duplicate,0,240,227,0.9458333333333332
Test/Not Eye/03c77ada4d.jpg,Not Eye,Train/Not Eye/3c22b84dd9.jpg,Not Eye,same-class near-duplicate,0,84,70,0.8333333333333334
Test/Not Eye/03e7ef7b66.jpg,Not Eye,Train/Not Eye/8b8ae57d04.jpg,Not Eye,same-class near-duplicate,0,289,275,0.9515570934256056
Test/Not Eye/0a10c239b1.jpg,Not Eye,Train/Not Eye/5fb78d2ed9.jpg,Not Eye,same-class near-duplicate,0,353,346,0.980169971671388
Test/Not Eye/0a3c6c1160.jpg,Not Eye,Train/Not Eye/5d28b2c144.jpg,Not Eye,same-class near-duplicate,0,242,238,0.9834710743801652
Test/Not Eye/0a6124179a.jpg,Not Eye,Train/Not Eye/5bc77b9729.jpg,Not Eye,same-class near-duplicate,0,289,241,0.8339100346020761
Test/Not Eye/0a6cd580bd.jpg,Not Eye,Train/Not Eye/5c1a35682e.jpg,Not Eye,same-class near-duplicate,0,266,260,0.9774436090225564
Test/Not Eye/0a813ee57a.jpg,Not Eye,Train/Not Eye/8d9b69ad1a.jpg,Not Eye,same-class near-duplicate,0,456,442,0.9692982456140352
Test/Not Eye/0a8356be04.jpg,Not Eye,Train/Not Eye/3cfc4e1517.jpg,Not Eye,same-class near-duplicate,0,103,96,0.9320388349514565
Test/Not Eye/0abcfb9143.jpg,Not Eye,Train/Not Eye/6b508a31f2.jpg,Not Eye,same-class near-duplicate,2,398,381,0.957286432160804
Test/Not Eye/0ac9b0c175.jpg,Not Eye,Train/Not Eye/5a9fd2c50e.jpg,Not Eye,same-class near-duplicate,0,341,324,0.9501466275659824
Test/Not Eye/0aff82ae15.jpg,Not Eye,Train/Not Eye/17bd5d7493.jpg,Not Eye,same-class near-duplicate,0,336,324,0.9642857142857144
Test/Not Eye/0b31c7c40f.jpg,Not Eye,Train/Not Eye/4ce0494bf8.jpg,Not Eye,same-class near-duplicate,0,158,152,0.9620253164556962
Test/Not Eye/0b5499899b.jpg,Not Eye,Train/Not Eye/5e0186349a.jpg,Not Eye,same-class near-duplicate,0,239,217,0.9079497907949792
Test/Not Eye/0b59806867.jpg,Not Eye,Train/Not Eye/5d633cebaa.jpg,Not Eye,same-class near-duplicate,0,253,233,0.9209486166007904
Test/Not Eye/0b6b26d00a.jpg,Not Eye,Train/Not Eye/3e0d94025c.jpg,Not Eye,same-class near-duplicate,0,310,304,0.9806451612903224
Test/Not Eye/0b8b93a61c.jpg,Not Eye,Train/Not Eye/5b86c2854e.jpg,Not Eye,same-class near-duplicate,0,379,375,0.9894459102902374
Test/Not Eye/0b97c99f59.jpg,Not Eye,Train/Not Eye/3c84676bd2.jpg,Not Eye,same-class near-duplicate,0,352,342,0.9715909090909092
Test/Not Eye/0ba82d276f.jpg,Not Eye,Train/Not Eye/7b2ce7bdb5.jpg,Not Eye,same-class near-duplicate,0,340,326,0.9588235294117649
Test/Not Eye/0be07dfaaf.jpg,Not Eye,Train/Not Eye/7c6e41f278.jpg,Not Eye,same-class near-duplicate,0,171,168,0.9824561403508772
Test/Not Eye/0be899ea05.jpg,Not Eye,Train/Not Eye/85acddfe82.jpg,Not Eye,same-class near-duplicate,12,189,182,0.9629629629629628
Test/Not Eye/0c245ddd6a.jpg,Not Eye,Train/Not Eye/9cd01fcd31.jpg,Not Eye,same-class near-duplicate,0,378,367,0.9708994708994708
Test/Not Eye/0c3fc00453.jpg,Not Eye,Train/Not Eye/7fa691b17b.jpg,Not Eye,same-class near-duplicate,0,405,405,1.0
Test/Not Eye/0c6c94660d.jpg,Not Eye,Train/Not Eye/13d2fb8a1b.jpg,Not Eye,same-class near-duplicate,2,425,419,0.9858823529411764
Test/Not Eye/0c74fbfe81.jpg,Not Eye,Train/Not Eye/7bb1e60805.jpg,Not Eye,same-class near-duplicate,0,300,295,0.9833333333333332
Test/Not Eye/0ca1957d08.jpg,Not Eye,Train/Not Eye/21d8403677.jpg,Not Eye,same-class near-duplicate,0,289,281,0.972318339100346
Test/Not Eye/0d09b0e580.jpg,Not Eye,Train/Not Eye/9d7f91c812.jpg,Not Eye,same-class near-duplicate,0,213,210,0.9859154929577464
Test/Not Eye/0d5934c096.jpg,Not Eye,Train/Not Eye/28f881b27f.jpg,Not Eye,same-class near-duplicate,0,287,259,0.902439024390244
Test/Not Eye/0d63dd091d.jpg,Not Eye,Train/Not Eye/9c928f4aac.jpg,Not Eye,same-class near-duplicate,0,335,324,0.9671641791044776
Test/Not Eye/0d6461bb49.jpg,Not Eye,Train/Not Eye/3c7a9f5243.jpg,Not Eye,same-class near-duplicate,0,246,234,0.951219512195122
Test/Not Eye/0d9b22e84f.jpg,Not Eye,Train/Not Eye/3b4e9e7b19.jpg,Not Eye,same-class near-duplicate,0,356,353,0.9915730337078652
Test/Not Eye/0db0e54368.jpg,Not Eye,Train/Not Eye/7c6ffb61ba.jpg,Not Eye,same-class near-duplicate,0,229,217,0.9475982532751092
Test/Not Eye/0dd73a4ca5.jpg,Not Eye,Train/Not Eye/13a52289d5.jpg,Not Eye,same-class near-duplicate,0,321,304,0.9470404984423676
Test/Not Eye/0dfaa738ba.jpg,Not Eye,Train/Not Eye/3bb19fd58f.jpg,Not Eye,same-class near-duplicate,0,330,327,0.990909090909091
Test/Not Eye/0e188c151b.jpg,Not Eye,Train/Not Eye/3eeba485b4.jpg,Not Eye,same-class near-duplicate,0,353,350,0.9915014164305948
Test/Not Eye/0e54a8f638.jpg,Not Eye,Train/Not Eye/8f6fe716ca.jpg,Not Eye,same-class near-duplicate,0,221,218,0.986425339366516
Test/Not Eye/0e661d4b4d.jpg,Not Eye,Train/Not Eye/09afaaeef2.jpg,Not Eye,same-class near-duplicate,2,265,260,0.981132075471698
Test/Not Eye/0ee9c5e22d.jpg,Not Eye,Train/Not Eye/3ca042da1d.jpg,Not Eye,same-class near-duplicate,0,367,359,0.9782016348773842
Test/Not Eye/0ef0b97946.jpg,Not Eye,Train/Not Eye/56eada35bb.jpg,Not Eye,same-class near-duplicate,0,358,348,0.9720670391061452
Test/Not Eye/0ef516b382.jpg,Not Eye,Train/Not Eye/24f4cfb13a.jpg,Not Eye,same-class near-duplicate,0,253,251,0.9920948616600792
Test/Not Eye/0f0b55a701.jpg,Not Eye,Train/Not Eye/4c99a589fe.jpg,Not Eye,same-class near-duplicate,0,207,191,0.9227053140096618
Test/Not Eye/0f19524f35.jpg,Not Eye,Train/Not Eye/6b6d07a739.jpg,Not Eye,same-class near-duplicate,0,291,254,0.872852233676976
Test/Not Eye/0f67dc615e.jpg,Not Eye,Train/Not Eye/3d6b378fdb.jpg,Not Eye,same-class near-duplicate,2,251,248,0.9880478087649402
Test/Not Eye/0f8d1a4fca.jpg,Not Eye,Train/Not Eye/3ae6cd47da.jpg,Not Eye,same-class near-duplicate,0,262,259,0.9885496183206108
Test/Not Eye/0ff068e0ea.jpg,Not Eye,Train/Not Eye/56cac5b702.jpg,Not Eye,same-class near-duplicate,0,323,300,0.9287925696594428
Test/Not Eye/1a184604a9.jpg,Not Eye,Train/Not Eye/3ad980b35b.jpg,Not Eye,same-class near-duplicate,0,225,220,0.9777777777777776
Test/Not Eye/1a1920e559.jpg,Not Eye,Train/Not Eye/26af243f03.jpg,Not Eye,same-class near-duplicate,0,342,324,0.9473684210526316
Test/Not Eye/1a21cfe891.jpg,Not Eye,Train/Not Eye/7c92a8be3b.jpg,Not Eye,same-class near-duplicate,0,465,462,0.9935483870967742
Test/Not Eye/1a5c72b3f6.jpg,Not Eye,Train/Not Eye/8a9cd7728f.jpg,Not Eye,same-class near-duplicate,0,332,319,0.960843373493976
Test/Not Eye/1a63aeb2fd.jpg,Not Eye,Train/Not Eye/8fbcf699e6.jpg,Not Eye,same-class near-duplicate,2,396,387,0.9772727272727272
Test/Not Eye/1a673018f4.jpg,Not Eye,Train/Not Eye/6db53b9ed6.jpg,Not Eye,same-class near-duplicate,2,380,371,0.9763157894736842
Test/Not Eye/1a6dfec388.jpg,Not Eye,Train/Not Eye/5ab806bfd6.jpg,Not Eye,same-class near-duplicate,2,206,190,0.9223300970873788
Test/Not Eye/1aa0bfd414.jpg,Not Eye,Train/Not Eye/9dcac3cc68.jpg,Not Eye,same-class near-duplicate,0,322,318,0.9875776397515528
Test/Not Eye/1aa51cd278.jpg,Not Eye,Train/Not Eye/6b74664f79.jpg,Not Eye,same-class near-duplicate,0,351,339,0.9658119658119658
Test/Not Eye/1aa843916f.jpg,Not Eye,Train/Not Eye/3de4594be1.jpg,Not Eye,same-class near-duplicate,0,206,170,0.8252427184466019
Test/Not Eye/1ab41e982a.jpg,Not Eye,Train/Not Eye/05bc964db3.jpg,Not Eye,same-class near-duplicate,0,329,321,0.9756838905775076
Test/Not Eye/1adc11b705.jpg,Not Eye,Train/Not Eye/8d086cf874.jpg,Not Eye,same-class near-duplicate,0,236,224,0.9491525423728814
Test/Not Eye/1b18f801a2.jpg,Not Eye,Train/Not Eye/5b4b01279d.jpg,Not Eye,same-class near-duplicate,0,194,186,0.9587628865979382
Test/Not Eye/1b1da65165.jpg,Not Eye,Train/Not Eye/8b30ccaf83.jpg,Not Eye,same-class near-duplicate,0,306,301,0.9836601307189542
Test/Not Eye/1b2103924c.jpg,Not Eye,Train/Not Eye/3d9850d0cf.jpg,Not Eye,same-class near-duplicate,0,376,371,0.9867021276595744
Test/Not Eye/1b2b7b9df9.jpg,Not Eye,Train/Not Eye/4c2cae1e23.jpg,Not Eye,same-class near-duplicate,0,316,310,0.981012658227848
Test/Not Eye/1b67a0fdb7.jpg,Not Eye,Train/Not Eye/9bb0ceee7d.jpg,Not Eye,same-class near-duplicate,2,241,236,0.979253112033195
Test/Not Eye/1b6860747f.jpg,Not Eye,Train/Not Eye/16f44392a2.jpg,Not Eye,same-class near-duplicate,2,206,196,0.9514563106796116
Test/Not Eye/1b6e9349e2.jpg,Not Eye,Train/Not Eye/21ca38c2a8.jpg,Not Eye,same-class near-duplicate,0,267,255,0.9550561797752808
Test/Not Eye/1bdfc278e0.jpg,Not Eye,Train/Not Eye/6b33eb3115(1).jpg,Not Eye,same-class near-duplicate,2,322,271,0.8416149068322981
Test/Not Eye/1c158cb8ed.jpg,Not Eye,Train/Not Eye/6f51765c8b.jpg,Not Eye,same-class near-duplicate,0,197,168,0.8527918781725888
Test/Not Eye/1c1d43cdff.jpg,Not Eye,Train/Not Eye/8e9df8b6b8.jpg,Not Eye,same-class near-duplicate,0,321,310,0.9657320872274144
Test/Not Eye/1c7db700dd.jpg,Not Eye,Train/Not Eye/39b872b21c.jpg,Not Eye,same-class near-duplicate,0,272,264,0.9705882352941176
Test/Not Eye/1c84507f90.jpg,Not Eye,Train/Not Eye/205e0d3c63.jpg,Not Eye,same-class near-duplicate,0,346,343,0.9913294797687862
Test/Not Eye/1c90b2be0c.jpg,Not Eye,Train/Not Eye/7b2ff46eeb.jpg,Not Eye,same-class near-duplicate,0,233,225,0.965665236051502
Test/Not Eye/1caf88408f.jpg,Not Eye,Train/Not Eye/95c49fa139.jpg,Not Eye,same-class near-duplicate,0,309,294,0.9514563106796116
Test/Not Eye/1cbf331f74.jpg,Not Eye,Train/Not Eye/3a4906810d.jpg,Not Eye,same-class near-duplicate,0,191,181,0.9476439790575916
Test/Not Eye/1cc84b7153.jpg,Not Eye,Train/Not Eye/22d4087443.jpg,Not Eye,same-class near-duplicate,0,318,302,0.949685534591195
Test/Not Eye/1cfec0c132.jpg,Not Eye,Train/Not Eye/9d7b9130a9.jpg,Not Eye,same-class near-duplicate,0,131,102,0.7786259541984732
Test/Not Eye/1d2997749b.jpg,Not Eye,Train/Not Eye/6d07675432.jpg,Not Eye,same-class near-duplicate,0,243,242,0.9958847736625516
Test/Not Eye/1d57fc1de0.jpg,Not Eye,Train/Not Eye/4cbc198f38.jpg,Not Eye,same-class near-duplicate,0,418,416,0.9952153110047848
Test/Not Eye/1d601d6f96.jpg,Not Eye,Train/Not Eye/12f284ec5d.jpg,Not Eye,same-class near-duplicate,0,318,314,0.9874213836477987
Test/Not Eye/1d81eaece2.jpg,Not Eye,Train/Not Eye/10b105bbcf.jpg,Not Eye,same-class near-duplicate,0,190,183,0.963157894736842
Test/Not Eye/1d9a85039f.jpg,Not Eye,Train/Not Eye/97fc375b9f.jpg,Not Eye,same-class near-duplicate,0,271,261,0.96309963099631
Test/Not Eye/1da4eb7584.jpg,Not Eye,Train/Not Eye/7eeef8f3bd.jpg,Not Eye,same-class near-duplicate,0,141,128,0.9078014184397164
Test/Not Eye/1dc527442b.jpg,Not Eye,Train/Not Eye/10ea951e2c.jpg,Not Eye,same-class near-duplicate,0,264,252,0.9545454545454546
Test/Not Eye/1e3f3359d5.jpg,Not Eye,Train/Not Eye/3d24434b7a.jpg,Not Eye,same-class near-duplicate,0,161,152,0.9440993788819876
Test/Not Eye/1e5a98ae55.jpg,Not Eye,Train/Not Eye/786938fece.jpg,Not Eye,same-class near-duplicate,2,250,245,0.98
Test/Not Eye/1e5fa10888.jpg,Not Eye,Train/Not Eye/19a7b94e11.jpg,Not Eye,same-class near-duplicate,0,260,256,0.9846153846153848
Test/Not Eye/1e5fb88200.jpg,Not Eye,Train/Not Eye/6c7fafdc32.jpg,Not Eye,same-class near-duplicate,2,432,414,0.9583333333333334
Test/Not Eye/1e6623ddfb.jpg,Not Eye,Train/Not Eye/4a6fadbe8f.jpg,Not Eye,same-class near-duplicate,0,145,136,0.9379310344827586
Test/Not Eye/1e820ca3b7.jpg,Not Eye,Train/Not Eye/7bd0ee7d11.jpg,Not Eye,same-class near-duplicate,0,390,369,0.946153846153846
Test/Not Eye/1eb5189a9a.jpg,Not Eye,Train/Not Eye/4e60c51db6.jpg,Not Eye,same-class near-duplicate,0,215,198,0.9209302325581395
Test/Not Eye/1ebaf95df7.jpg,Not Eye,Train/Not Eye/7e03780b90.jpg,Not Eye,same-class near-duplicate,0,250,240,0.96
Test/Not Eye/1ebbd7ab86.jpg,Not Eye,Train/Not Eye/4d511b09f8.jpg,Not Eye,same-class near-duplicate,0,280,275,0.9821428571428572
Test/Not Eye/1ed0797326.jpg,Not Eye,Train/Not Eye/4eb72eb794.jpg,Not Eye,same-class near-duplicate,0,135,118,0.8740740740740741
Test/Not Eye/1eea29a7b3.jpg,Not Eye,Train/Not Eye/6f4f3f84a6.jpg,Not Eye,same-class near-duplicate,0,147,142,0.9659863945578232
Test/Not Eye/1f09ecf262.jpg,Not Eye,Train/Not Eye/5ad4fa50b8.jpg,Not Eye,same-class near-duplicate,0,280,272,0.9714285714285714
Test/Not Eye/1f4cefdaf0.jpg,Not Eye,Train/Not Eye/3ce7701ebd.jpg,Not Eye,same-class near-duplicate,0,162,143,0.8827160493827161
Test/Not Eye/1f57605b9c.jpg,Not Eye,Train/Not Eye/4b583504d9.jpg,Not Eye,same-class near-duplicate,0,80,80,1.0
Test/Not Eye/1f6703e926.jpg,Not Eye,Train/Not Eye/8f52854785.jpg,Not Eye,same-class near-duplicate,0,469,461,0.9829424307036247
Test/Not Eye/1f71179ba6.jpg,Not Eye,Train/Not Eye/23f644af28.jpg,Not Eye,same-class near-duplicate,0,397,390,0.982367758186398
Test/Not Eye/1f7d159f91.jpg,Not Eye,Train/Not Eye/29a306fa6e.jpg,Not Eye,same-class near-duplicate,0,343,341,0.9941690962099126
Test/Not Eye/1f976e9337.jpg,Not Eye,Train/Not Eye/5c92a8dacc.jpg,Not Eye,same-class near-duplicate,0,237,223,0.940928270042194
Test/Not Eye/1fa7404c7f.jpg,Not Eye,Train/Not Eye/8de570fe3d.jpg,Not Eye,same-class near-duplicate,0,200,196,0.98
Test/Not Eye/1fa9069a37.jpg,Not Eye,Train/Not Eye/06d8a495b7.jpg,Not Eye,same-class near-duplicate,0,405,404,0.9975308641975308
Test/Not Eye/1fcc32417a.jpg,Not Eye,Train/Not Eye/571ab4e8a2.jpg,Not Eye,same-class near-duplicate,0,303,295,0.9735973597359736
Test/Not Eye/1ff3ffa923.jpg,Not Eye,Train/Not Eye/07b1fd2bea.jpg,Not Eye,same-class near-duplicate,2,420,411,0.9785714285714284
Test/Not Eye/2a3fea2107.jpg,Not Eye,Train/Not Eye/7ecbf8df97.jpg,Not Eye,same-class near-duplicate,0,217,178,0.8202764976958525
Test/Not Eye/2a6090151a.jpg,Not Eye,Train/Not Eye/7357794aa9.jpg,Not Eye,same-class near-duplicate,0,255,249,0.976470588235294
Test/Not Eye/2a667719d6.jpg,Not Eye,Train/Not Eye/4dd5ccddce.jpg,Not Eye,same-class near-duplicate,0,234,232,0.9914529914529916
Test/Not Eye/2a93afc49b.jpg,Not Eye,Train/Not Eye/017d206bca.jpg,Not Eye,same-class near-duplicate,0,265,251,0.9471698113207548
Test/Not Eye/2b17bb86c5.jpg,Not Eye,Train/Not Eye/23ecb0b167.jpg,Not Eye,same-class near-duplicate,0,225,221,0.9822222222222222
Test/Not Eye/2b3d210c90.jpg,Not Eye,Train/Not Eye/3d148b6ceb.jpg,Not Eye,same-class near-duplicate,0,460,456,0.991304347826087
Test/Not Eye/2b7df121fa.jpg,Not Eye,Train/Not Eye/07c9d4f7f6.jpg,Not Eye,same-class near-duplicate,0,277,257,0.927797833935018
Test/Not Eye/2b805d2a64.jpg,Not Eye,Train/Not Eye/3e45e8aa09.jpg,Not Eye,same-class near-duplicate,0,314,300,0.9554140127388536
Test/Not Eye/2b85be32d2.jpg,Not Eye,Train/Not Eye/9f1bea68a0.jpg,Not Eye,same-class near-duplicate,0,267,255,0.9550561797752808
Test/Not Eye/2bacf1d55a.jpg,Not Eye,Train/Not Eye/5bf954ae7f.jpg,Not Eye,same-class near-duplicate,2,180,170,0.9444444444444444
Test/Not Eye/2bba91f416.jpg,Not Eye,Train/Not Eye/36db286bbc.jpg,Not Eye,same-class near-duplicate,0,385,379,0.9844155844155844
Test/Not Eye/2bcf427424.jpg,Not Eye,Train/Not Eye/7bdfe4bb28.jpg,Not Eye,same-class near-duplicate,0,200,167,0.835
Test/Not Eye/2c4ffd1cd9.jpg,Not Eye,Train/Not Eye/4a17bf7777.jpg,Not Eye,same-class near-duplicate,0,304,294,0.9671052631578948
Test/Not Eye/2c915092ba.jpg,Not Eye,Train/Not Eye/4b70c10760.jpg,Not Eye,same-class near-duplicate,0,430,414,0.9627906976744186
Test/Not Eye/2c9b786e0a.jpg,Not Eye,Train/Not Eye/8a818620c3.jpg,Not Eye,same-class near-duplicate,0,373,373,1.0
Test/Not Eye/2cd7b38062.jpg,Not Eye,Train/Not Eye/8a358db06c.jpg,Not Eye,same-class near-duplicate,0,179,176,0.9832402234636872
Test/Not Eye/2cd9643a66.jpg,Not Eye,Train/Not Eye/25e5fb87cc.jpg,Not Eye,same-class near-duplicate,2,343,341,0.9941690962099126
Test/Not Eye/2d314ad204.jpg,Not Eye,Train/Not Eye/3b263b7383.jpg,Not Eye,same-class near-duplicate,0,147,138,0.9387755102040816
Test/Not Eye/2d5ab32c96.jpg,Not Eye,Train/Not Eye/6e6123e767.jpg,Not Eye,same-class near-duplicate,0,269,263,0.9776951672862454
Test/Not Eye/2d9261bc9c.jpg,Not Eye,Train/Not Eye/69d01c3abe.jpg,Not Eye,same-class near-duplicate,0,256,236,0.921875
Test/Not Eye/2d9332be32.jpg,Not Eye,Train/Not Eye/034bb7c9c6.jpg,Not Eye,same-class near-duplicate,0,119,113,0.9495798319327732
Test/Not Eye/2d948fcff4.jpg,Not Eye,Train/Not Eye/10eb87f1ea.jpg,Not Eye,same-class near-duplicate,0,434,428,0.9861751152073732
Test/Not Eye/2da1332a02.jpg,Not Eye,Train/Not Eye/6d7ded0af2.jpg,Not Eye,same-class near-duplicate,0,318,310,0.9748427672955976
Test/Not Eye/2dff59f5e6.jpg,Not Eye,Train/Not Eye/8a12773f12.jpg,Not Eye,same-class near-duplicate,0,385,375,0.974025974025974
Test/Not Eye/2e1a012bb0.jpg,Not Eye,Train/Not Eye/5a29964f70.jpg,Not Eye,same-class near-duplicate,0,261,251,0.9616858237547892
Test/Not Eye/2e21f1fa64.jpg,Not Eye,Train/Not Eye/28a41c7f83.jpg,Not Eye,same-class near-duplicate,0,250,250,1.0
Test/Not Eye/2e2b1fc3ef.jpg,Not Eye,Train/Not Eye/7c657fc3fc.jpg,Not Eye,same-class near-duplicate,2,359,330,0.9192200557103064
Test/Not Eye/2e34c2526f.jpg,Not Eye,Train/Not Eye/8d781400d3.jpg,Not Eye,same-class near-duplicate,0,329,326,0.9908814589665652
Test/Not Eye/2e89a3158d.jpg,Not Eye,Train/Not Eye/3e7c8dd7d1.jpg,Not Eye,same-class near-duplicate,0,264,262,0.9924242424242424
Test/Not Eye/2e9b9ec2a7.jpg,Not Eye,Train/Not Eye/58d9036046.jpg,Not Eye,same-class near-duplicate,0,315,311,0.9873015873015872
Test/Not Eye/2eb4317ed6.jpg,Not Eye,Train/Not Eye/56c2f31417.jpg,Not Eye,same-class near-duplicate,2,209,208,0.9952153110047848
Test/Not Eye/2f0ef14a4b.jpg,Not Eye,Train/Not Eye/63d22b4a1d.jpg,Not Eye,same-class near-duplicate,0,396,388,0.9797979797979798
Test/Not Eye/2f2b76aafa.jpg,Not Eye,Train/Not Eye/38fb60ab5c.jpg,Not Eye,same-class near-duplicate,2,129,118,0.9147286821705426
Test/Not Eye/2f2f55df75.jpg,Not Eye,Train/Not Eye/8aa5c06d27.jpg,Not Eye,same-class near-duplicate,0,359,352,0.98050139275766
Test/Not Eye/2f48f0a08e.jpg,Not Eye,Train/Not Eye/9cfbbc24fa.jpg,Not Eye,same-class near-duplicate,2,318,313,0.9842767295597484
Test/Not Eye/2f4a394175.jpg,Not Eye,Train/Not Eye/8a9500f879.jpg,Not Eye,same-class near-duplicate,0,319,310,0.9717868338557992
Test/Not Eye/2f6b53cb64.jpg,Not Eye,Train/Not Eye/7a79e7865f.jpg,Not Eye,same-class near-duplicate,0,136,129,0.9485294117647058
Test/Not Eye/2f6cc87911.jpg,Not Eye,Train/Not Eye/3d7b2d49d0.jpg,Not Eye,same-class near-duplicate,0,256,231,0.90234375
Test/Not Eye/2f8013eec0.jpg,Not Eye,Train/Not Eye/8fc7f43930.jpg,Not Eye,same-class near-duplicate,0,322,319,0.9906832298136646
Test/Not Eye/2fb4fd0770.jpg,Not Eye,Train/Not Eye/4f5db1ab21.jpg,Not Eye,same-class near-duplicate,0,285,279,0.9789473684210528
Test/Not Eye/2fff4054b6.jpg,Not Eye,Train/Not Eye/3b11e111b7.jpg,Not Eye,same-class near-duplicate,0,167,161,0.9640718562874252
Test/Not Eye/2fff8c93fa.jpg,Not Eye,Train/Not Eye/54ff298558.jpg,Not Eye,same-class near-duplicate,0,153,149,0.9738562091503268
Test/Not Eye/3a16f237fc.jpg,Not Eye,Train/Not Eye/5fa32d6e4a.jpg,Not Eye,same-class near-duplicate,0,321,317,0.9875389408099688
Test/Not Eye/3a27e58575.jpg,Not Eye,Train/Not Eye/43bdfd0cdc.jpg,Not Eye,same-class near-duplicate,2,515,504,0.9786407766990292
Test/Not Eye/3a8cff5201.jpg,Not Eye,Train/Not Eye/5aafed78a1.jpg,Not Eye,same-class near-duplicate,2,282,274,0.9716312056737588
'''
EXACT_CSV = r'''sha256,split,class,path,group_size
23a580dc3aabd33d0d7affe4c4198aa707efe60c57fae84e2efdbee9683172d4,Train,Not Eye,Train/Not Eye/6b1f9f0434(1).jpg,3
23a580dc3aabd33d0d7affe4c4198aa707efe60c57fae84e2efdbee9683172d4,Train,Not Eye,Train/Not Eye/6b1f9f0434(2).jpg,3
23a580dc3aabd33d0d7affe4c4198aa707efe60c57fae84e2efdbee9683172d4,Train,Not Eye,Train/Not Eye/6b1f9f0434.jpg,3
254d6d28bd4e6327d74ba738fcfa21aa89012d367289a9a567cc9c1c6763d0ee,Train,Not Eye,Train/Not Eye/6cf07df097(1).jpg,2
254d6d28bd4e6327d74ba738fcfa21aa89012d367289a9a567cc9c1c6763d0ee,Train,Not Eye,Train/Not Eye/6cf07df097.jpg,2
28f15a2fdd2cee254fff3eb5170b8952f64b1c88578eef870c1459a1626f06da,Train,Not Eye,Train/Not Eye/6b1f4f32d0(1).jpg,3
28f15a2fdd2cee254fff3eb5170b8952f64b1c88578eef870c1459a1626f06da,Train,Not Eye,Train/Not Eye/6b1f4f32d0(2).jpg,3
28f15a2fdd2cee254fff3eb5170b8952f64b1c88578eef870c1459a1626f06da,Train,Not Eye,Train/Not Eye/6b1f4f32d0.jpg,3
370ef303d2d8d1467543b086cf526b3f03a53d15239670150a12ac97f9bb6feb,Train,Not Eye,Train/Not Eye/6b33eb3115(1).jpg,3
370ef303d2d8d1467543b086cf526b3f03a53d15239670150a12ac97f9bb6feb,Train,Not Eye,Train/Not Eye/6b33eb3115(2).jpg,3
370ef303d2d8d1467543b086cf526b3f03a53d15239670150a12ac97f9bb6feb,Train,Not Eye,Train/Not Eye/6b33eb3115.jpg,3
59ddd6d8adcbd2341417eec9c2da9b183b3a893485bed4ee95c2d045f43bb3ac,Train,Not Eye,Train/Not Eye/6b2fcd6a8b(1).jpg,3
59ddd6d8adcbd2341417eec9c2da9b183b3a893485bed4ee95c2d045f43bb3ac,Train,Not Eye,Train/Not Eye/6b2fcd6a8b(2).jpg,3
59ddd6d8adcbd2341417eec9c2da9b183b3a893485bed4ee95c2d045f43bb3ac,Train,Not Eye,Train/Not Eye/6b2fcd6a8b.jpg,3
7610cf0f3b4dbf195bcfab0b402169402bd3a6de5c1b4d501ac1ae4ef148ebef,Train,Not Eye,Train/Not Eye/6ceb547ab5(1).jpg,2
7610cf0f3b4dbf195bcfab0b402169402bd3a6de5c1b4d501ac1ae4ef148ebef,Train,Not Eye,Train/Not Eye/6ceb547ab5.jpg,2
78654f9647dab68a0d7f5ed37b91a8da5588c74f9f00f06b8c4780d2b087ef8e,Train,Not Eye,Train/Not Eye/6b3c70f9e4(1).jpg,3
78654f9647dab68a0d7f5ed37b91a8da5588c74f9f00f06b8c4780d2b087ef8e,Train,Not Eye,Train/Not Eye/6b3c70f9e4(2).jpg,3
78654f9647dab68a0d7f5ed37b91a8da5588c74f9f00f06b8c4780d2b087ef8e,Train,Not Eye,Train/Not Eye/6b3c70f9e4.jpg,3
89f29792e89b1825c99177585e2f13ddfac3e3796e5be30561876d16aa9b6bbf,Train,Not Eye,Train/Not Eye/6b2468cb2e(1).jpg,3
89f29792e89b1825c99177585e2f13ddfac3e3796e5be30561876d16aa9b6bbf,Train,Not Eye,Train/Not Eye/6b2468cb2e(2).jpg,3
89f29792e89b1825c99177585e2f13ddfac3e3796e5be30561876d16aa9b6bbf,Train,Not Eye,Train/Not Eye/6b2468cb2e.jpg,3
8feb0b4b01e9d16db22333ccec4d9cdbdc5377f1a556a0747a26479f640aa8bd,Train,Not Eye,Train/Not Eye/6b2a5b9f02(1).jpg,3
8feb0b4b01e9d16db22333ccec4d9cdbdc5377f1a556a0747a26479f640aa8bd,Train,Not Eye,Train/Not Eye/6b2a5b9f02(2).jpg,3
8feb0b4b01e9d16db22333ccec4d9cdbdc5377f1a556a0747a26479f640aa8bd,Train,Not Eye,Train/Not Eye/6b2a5b9f02.jpg,3
90af79a44cab55d04b8e7672b2bb9cab54e65725ec9abae8160a43a0ba4228c1,Train,Not Eye,Train/Not Eye/6b243b6acc(1).jpg,3
90af79a44cab55d04b8e7672b2bb9cab54e65725ec9abae8160a43a0ba4228c1,Train,Not Eye,Train/Not Eye/6b243b6acc(2).jpg,3
90af79a44cab55d04b8e7672b2bb9cab54e65725ec9abae8160a43a0ba4228c1,Train,Not Eye,Train/Not Eye/6b243b6acc.jpg,3
988394dab93894795bbf0c318292b522b7cf572bf413d393881f780c1f2a70ab,Train,Not Eye,Train/Not Eye/6cd666a8bf(1).jpg,2
988394dab93894795bbf0c318292b522b7cf572bf413d393881f780c1f2a70ab,Train,Not Eye,Train/Not Eye/6cd666a8bf.jpg,2
a523acc4f123f8ec8a5a49d162e2284e9b5a64e3bb15b661a9c8d0487bfa9a6c,Train,Not Eye,Train/Not Eye/6b37b3d6d9(1).jpg,3
a523acc4f123f8ec8a5a49d162e2284e9b5a64e3bb15b661a9c8d0487bfa9a6c,Train,Not Eye,Train/Not Eye/6b37b3d6d9(2).jpg,3
a523acc4f123f8ec8a5a49d162e2284e9b5a64e3bb15b661a9c8d0487bfa9a6c,Train,Not Eye,Train/Not Eye/6b37b3d6d9.jpg,3
be71b8a93bd667af157215caa1b7784c0b3e2078e3c97ac80a66ac0d08745a37,Train,Not Eye,Train/Not Eye/6ce2e648ee(1).jpg,2
be71b8a93bd667af157215caa1b7784c0b3e2078e3c97ac80a66ac0d08745a37,Train,Not Eye,Train/Not Eye/6ce2e648ee.jpg,2
c06597f23f4b663cf5063a20646da03d6f43976fcf745e7c77eb6990700db82b,Train,Not Eye,Train/Not Eye/6cdcda7b77(1).jpg,2
c06597f23f4b663cf5063a20646da03d6f43976fcf745e7c77eb6990700db82b,Train,Not Eye,Train/Not Eye/6cdcda7b77.jpg,2
f9f772c89822220c81cec8746f23ab66e0b09eda0d34fe3159d5ff947454e83a,Train,Not Eye,Train/Not Eye/6b30121ed8(1).jpg,3
f9f772c89822220c81cec8746f23ab66e0b09eda0d34fe3159d5ff947454e83a,Train,Not Eye,Train/Not Eye/6b30121ed8(2).jpg,3
f9f772c89822220c81cec8746f23ab66e0b09eda0d34fe3159d5ff947454e83a,Train,Not Eye,Train/Not Eye/6b30121ed8.jpg,3
'''

near = pd.read_csv(io.StringIO(NEAR_CSV))
exact = pd.read_csv(io.StringIO(EXACT_CSV))

print('High-confidence Train/Test relation rows:', len(near))
print('Unique affected old Test images:', near['test_path'].nunique())
print('Exact-duplicate membership rows:', len(exact))
display(near.head())

High-confidence Train/Test relation rows: 383
Unique affected old Test images: 382
Exact-duplicate membership rows: 40


,test_path,test_class,matched_train_path,matched_train_class,match_type,phash_distance,sift_good,sift_inliers,sift_inlier_ratio
0,Test/Cataract/cat_0_1007.jpg,Cataract,Train/Cataract/cat_0_8343.jpg,Cataract,same-class near-duplicate,8,73,72,0.986301
1,Test/Cataract/cat_0_1022.jpg,Cataract,Train/Cataract/cat_0_5245.jpg,Cataract,same-class near-duplicate,12,45,43,0.955556
2,Test/Cataract/cat_0_1065.jpg,Cataract,Train/Cataract/cat_0_2231.jpg,Cataract,same-class near-duplicate,12,232,223,0.961207
3,Test/Cataract/cat_0_1073.jpg,Cataract,Train/Cataract/cat_0_9852.jpg,Cataract,same-class near-duplicate,12,77,68,0.883117
4,Test/Cataract/cat_0_1074.jpg,Cataract,Train/Cataract/cat_0_9877.jpg,Cataract,same-class near-duplicate,10,82,79,0.963415


In [3]:
# Inventory the ORIGINAL dataset.
CLASS_ORDER = ['Cataract', 'Normal', 'Not Eye']
valid_ext = {'.jpg','.jpeg','.png','.bmp','.webp'}

rows = []
for split in ['Train','Test']:
    for cls in CLASS_ORDER:
        d = SOURCE / split / cls
        assert d.exists(), f'Missing folder: {d}'
        for p in sorted(d.rglob('*')):
            if p.is_file() and p.suffix.lower() in valid_ext:
                rel = p.relative_to(SOURCE).as_posix()
                rows.append({
                    'old_path': rel,
                    'old_split': split,
                    'class': cls,
                    'filename': p.name,
                    'source_abs': str(p)
                })
files = pd.DataFrame(rows)
print(files.groupby(['old_split','class']).size())
print('TOTAL:', len(files))

old_split  class   
Test       Cataract     800
           Normal       800
           Not Eye      952
Train      Cataract    3714
           Normal      4354
           Not Eye     3049
dtype: int64
TOTAL: 13669


## Build image families

Every confirmed near-duplicate pair is placed in the same connected component.
Exact duplicates are also connected.

If a connected component contains both **Cataract** and **Normal** labels (or otherwise contains multiple classes),
the whole component is quarantined rather than guessing the correct medical label.

In [4]:
class DSU:
    def __init__(self, items):
        self.p = {x:x for x in items}
        self.r = {x:0 for x in items}
    def find(self,x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self,a,b):
        if a not in self.p or b not in self.p: return
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]: self.r[ra] += 1

all_paths = files['old_path'].tolist()
dsu = DSU(all_paths)

# Confirmed high-confidence Train/Test pairs
for _, r in near.iterrows():
    dsu.union(r['test_path'], r['matched_train_path'])

# Exact duplicate groups
for sha, g in exact.groupby('sha256'):
    paths = [x for x in g['path'].tolist() if x in dsu.p]
    for x in paths[1:]:
        dsu.union(paths[0], x)

files['family_root'] = files['old_path'].map(dsu.find)

# Assign compact family IDs
roots = {r:i for i,r in enumerate(sorted(files['family_root'].unique()), start=1)}
files['family_id'] = files['family_root'].map(lambda x: f'F{roots[x]:05d}')

family_classes = files.groupby('family_id')['class'].agg(lambda x: sorted(set(x))).to_dict()
conflicted = {fid for fid, cs in family_classes.items() if len(cs) > 1}

files['label_conflict'] = files['family_id'].isin(conflicted)
print('Total families:', files['family_id'].nunique())
print('Cross-class conflicted families:', len(conflicted))
display(files[files.label_conflict].sort_values(['family_id','class','old_path']).head(50))

Total families: 13261
Cross-class conflicted families: 2


,old_path,old_split,class,filename,source_abs,family_root,family_id,label_conflict
11280,Test/Cataract/cat_0_1328.jpg,Test,Cataract,cat_0_1328.jpg,/content/drive/MyDrive/Cataract/Data/Test/Cata...,Test/Cataract/cat_0_1328.jpg,F00164,True
1124,Train/Cataract/cat_0_4270.jpg,Train,Cataract,cat_0_4270.jpg,/content/drive/MyDrive/Cataract/Data/Train/Cat...,Test/Cataract/cat_0_1328.jpg,F00164,True
5321,Train/Normal/cat_0_4602.jpg,Train,Normal,cat_0_4602.jpg,/content/drive/MyDrive/Cataract/Data/Train/Nor...,Test/Cataract/cat_0_1328.jpg,F00164,True
11404,Test/Cataract/cat_0_1566.jpg,Test,Cataract,cat_0_1566.jpg,/content/drive/MyDrive/Cataract/Data/Test/Cata...,Test/Cataract/cat_0_1566.jpg,F00288,True
5999,Train/Normal/cat_0_5885.jpg,Train,Normal,cat_0_5885.jpg,/content/drive/MyDrive/Cataract/Data/Train/Nor...,Test/Cataract/cat_0_1566.jpg,F00288,True


In [5]:
# Also deduplicate byte-identical files within the dataset by keeping one representative per SHA group.
# This only affects the exact duplicate groups already identified by the audit.
drop_exact_paths = set()
for sha, g in exact.groupby('sha256'):
    paths = sorted([x for x in g['path'].tolist() if x in set(files.old_path)])
    if len(paths) > 1:
        drop_exact_paths.update(paths[1:])

files['drop_exact_duplicate'] = files['old_path'].isin(drop_exact_paths)
print('Exact duplicate files to omit (keeping one representative/group):', files.drop_exact_duplicate.sum())

Exact duplicate files to omit (keeping one representative/group): 25


## Create family-level Train / Validation / Test split

We use approximately the same overall proportions implied by the original experiment:

- Train: **65%**
- Validation: **16%**
- Test: **19%**

The important rule is that an entire family goes to only one partition.

The split is stratified separately for each class.

In [6]:
eligible = files[~files.label_conflict & ~files.drop_exact_duplicate].copy()

# Family table; every non-conflicted family has exactly one class.
fam = eligible.groupby('family_id').agg(
    class_name=('class','first'),
    n_files=('old_path','size')
).reset_index()

TARGET = {'Train':0.65, 'Validation':0.16, 'Test':0.19}
assignment = {}

rng = np.random.default_rng(SEED)

for cls in CLASS_ORDER:
    fc = fam[fam.class_name == cls].copy()
    # randomized tie-breaking, then largest families first
    fc['rand'] = rng.random(len(fc))
    fc = fc.sort_values(['n_files','rand'], ascending=[False,True])

    total = int(fc.n_files.sum())
    targets = {k: TARGET[k]*total for k in TARGET}
    counts = {k:0 for k in TARGET}

    for _, r in fc.iterrows():
        # choose partition with largest remaining normalized deficit
        deficits = {}
        for k in TARGET:
            deficits[k] = (targets[k] - counts[k]) / max(targets[k], 1)
        dest = max(deficits, key=deficits.get)
        assignment[r.family_id] = dest
        counts[dest] += int(r.n_files)

    print('\n', cls, 'total=', total, 'targets=', {k:round(v) for k,v in targets.items()}, 'actual=', counts)

eligible['new_split'] = eligible['family_id'].map(assignment)
assert eligible['new_split'].notna().all()

display(eligible.groupby(['new_split','class']).size().unstack(fill_value=0))


 Cataract total= 4511 targets= {'Train': 2932, 'Validation': 722, 'Test': 857} actual= {'Train': 2932, 'Validation': 722, 'Test': 857}

 Normal total= 5152 targets= {'Train': 3349, 'Validation': 824, 'Test': 979} actual= {'Train': 3348, 'Validation': 825, 'Test': 979}

 Not Eye total= 3976 targets= {'Train': 2584, 'Validation': 636, 'Test': 755} actual= {'Train': 2584, 'Validation': 636, 'Test': 756}


class,Cataract,Normal,Not Eye
new_split,,,
Test,857,979,756
Train,2932,3348,2584
Validation,722,825,636


In [7]:
# Safety: create NEW folders only. Original SOURCE is untouched.
if OUTROOT.exists():
    print('WARNING: cleaned output already exists.')
    print('Delete it only if you intentionally want to rebuild the clean split.')
    raise RuntimeError(f'STOP: {OUTROOT} already exists. Rename/remove it intentionally before rerunning.')

for split in ['Train','Validation','Test']:
    for cls in CLASS_ORDER:
        (OUTROOT/split/cls).mkdir(parents=True, exist_ok=True)

# Quarantine conflicted families for evidence, not training.
if QUARANTINE.exists():
    QUARANTINE.mkdir(parents=True, exist_ok=True)

manifest_rows = []

for _, r in files.iterrows():
    src = Path(r.source_abs)
    if r.label_conflict:
        dest = QUARANTINE / r.family_id / r['class'] / r.filename
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
        status = 'QUARANTINED_LABEL_CONFLICT'
        new_split = 'Quarantine'
        new_path = str(dest)
    elif r.drop_exact_duplicate:
        status = 'OMITTED_EXACT_DUPLICATE'
        new_split = 'Omitted'
        new_path = ''
    else:
        new_split = assignment[r.family_id]
        dest = OUTROOT / new_split / r['class'] / r.filename
        # avoid name collision without changing content
        if dest.exists():
            dest = dest.with_name(f'{dest.stem}__{r.family_id}{dest.suffix}')
        shutil.copy2(src, dest)
        status = 'INCLUDED'
        new_path = str(dest)

    manifest_rows.append({
        'old_path': r.old_path,
        'old_split': r.old_split,
        'class': r['class'],
        'family_id': r.family_id,
        'new_split': new_split,
        'status': status,
        'new_path': new_path
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(REPORT_DIR/'split_manifest.csv', index=False)
print('Saved:', REPORT_DIR/'split_manifest.csv')
display(manifest.groupby(['status','class']).size().unstack(fill_value=0))

Saved: /content/drive/MyDrive/Cataract/FINAL_REVISION_2026_08/clean_split_audit/split_manifest.csv


class,Cataract,Normal,Not Eye
status,,,
INCLUDED,4511,5152,3976
OMITTED_EXACT_DUPLICATE,0,0,25
QUARANTINED_LABEL_CONFLICT,3,2,0


## Verification 1 — known high-confidence leakage pairs

Every previously confirmed Train/Test related pair must now either:
- be in the **same** new partition, or
- belong to a quarantined conflict family.

No previously confirmed pair is allowed to cross Train / Validation / Test.

In [8]:
m = manifest.set_index('old_path')
check = []
for _, r in near.iterrows():
    a = r.test_path
    b = r.matched_train_path
    if a not in m.index or b not in m.index:
        continue
    sa, sb = m.loc[a,'new_split'], m.loc[b,'new_split']
    bad = (
        sa in {'Train','Validation','Test'}
        and sb in {'Train','Validation','Test'}
        and sa != sb
    )
    check.append({
        'path_a':a,'path_b':b,
        'new_split_a':sa,'new_split_b':sb,
        'crosses_new_split':bad
    })

known_pair_check = pd.DataFrame(check)
known_pair_check.to_csv(REPORT_DIR/'known_pair_verification.csv', index=False)

n_bad = int(known_pair_check.crosses_new_split.sum())
print('Previously confirmed pairs crossing the NEW split:', n_bad)
assert n_bad == 0, 'STOP: known leakage pair still crosses the new split.'
print('PASS ✅')

Previously confirmed pairs crossing the NEW split: 0
PASS ✅


## Verification 2 — exact SHA-256 cross-partition duplicates

This independently hashes every copied image in the new dataset.

In [9]:
def sha256_file(p, chunk=1024*1024):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

hash_rows = []
for split in ['Train','Validation','Test']:
    for cls in CLASS_ORDER:
        for p in sorted((OUTROOT/split/cls).glob('*')):
            if p.is_file():
                hash_rows.append({
                    'split':split,'class':cls,'path':str(p.relative_to(OUTROOT)),
                    'sha256':sha256_file(p)
                })

hdf = pd.DataFrame(hash_rows)
cross = []
for sha,g in hdf.groupby('sha256'):
    splits = sorted(set(g.split))
    if len(splits) > 1:
        cross.append(g.assign(group_sha256=sha))

if cross:
    cross_df = pd.concat(cross, ignore_index=True)
else:
    cross_df = pd.DataFrame(columns=list(hdf.columns)+['group_sha256'])

cross_df.to_csv(REPORT_DIR/'exact_cross_split_duplicates_after_cleaning.csv', index=False)
print('Exact cross-partition duplicate rows:', len(cross_df))
assert len(cross_df) == 0, 'STOP: exact cross-partition duplicates remain.'
print('PASS ✅')

Exact cross-partition duplicate rows: 0
PASS ✅


## STOP / GO decision

If both checks above print **PASS**, Notebook 01 has completed the deterministic cleanup based on all already-confirmed leakage evidence.

**Before training**, run the separate verification script/notebook supplied in this package (`01B_Reaudit_Clean_Split.ipynb`) to perform a fresh perceptual/SIFT scan of the newly created split.

Do not train until that notebook also passes.

In [10]:
summary = {
    'source_total_files': int(len(files)),
    'included_total_files': int((manifest.status=='INCLUDED').sum()),
    'quarantined_label_conflict_files': int((manifest.status=='QUARANTINED_LABEL_CONFLICT').sum()),
    'omitted_exact_duplicate_files': int((manifest.status=='OMITTED_EXACT_DUPLICATE').sum()),
    'families_total': int(files.family_id.nunique()),
    'conflicted_families': int(len(conflicted)),
    'known_confirmed_pairs_crossing_new_split': int(n_bad),
    'exact_cross_split_duplicates_after_cleaning': int(len(cross_df)),
}
with open(REPORT_DIR/'clean_split_summary.json','w') as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2))
print('\nNEXT: run 01B_Reaudit_Clean_Split.ipynb')

{
  "source_total_files": 13669,
  "included_total_files": 13639,
  "quarantined_label_conflict_files": 5,
  "omitted_exact_duplicate_files": 25,
  "families_total": 13261,
  "conflicted_families": 2,
  "known_confirmed_pairs_crossing_new_split": 0,
  "exact_cross_split_duplicates_after_cleaning": 0
}

NEXT: run 01B_Reaudit_Clean_Split.ipynb
